<center><h1>zheng_yiyi_HW3</h1></center>
<br>
<br>

Name: yiyi zheng
<br>
Github Username: yiyizheng111
<br>
USC ID: 7092417109

## 1. Time Series Classification Part 1: Feature Creation/Extraction

### (a) Download Data

Package imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

Get the AReM Data Set

In [4]:
# Change this path to the location of AReM folder
DATA_DIR = Path("data/AReM")

activities = [
    "bending1",
    "bending2",
    "cycling",
    "lying",
    "sitting",
    "standing",
    "walking"
]

# Check whether all folders exist
for activity in activities:
    folder = DATA_DIR / activity
    print(activity, "->", folder.exists())

bending1 -> True
bending2 -> True
cycling -> True
lying -> True
sitting -> True
standing -> True
walking -> True


### (b) Test and Train Data

In [5]:
column_names = [
    "time",
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

def read_arem_file(file_path):
    rows = []

    with open(file_path, "r") as file:
        for line in file:
            if line.startswith("#"):
                continue

            # Handle both comma-separated and space-separated files
            line = line.strip().rstrip(",")
            values = line.replace(",", " ").split()
            rows.append([float(value) for value in values])

    return pd.DataFrame(rows, columns=column_names)


train_data = []
test_data = []

for activity in activities:
    folder = DATA_DIR / activity

    files = sorted(
        folder.glob("dataset*.csv"),
        key=lambda x: int(x.stem.replace("dataset", ""))
    )

    for file_path in files:
        dataset_number = int(file_path.stem.replace("dataset", ""))

        data = read_arem_file(file_path)

        instance = {
            "activity": activity,
            "dataset": dataset_number,
            "data": data
        }

        if activity in ["bending1", "bending2"]:
            if dataset_number in [1, 2]:
                test_data.append(instance)
            else:
                train_data.append(instance)
        else:
            if dataset_number in [1, 2, 3]:
                test_data.append(instance)
            else:
                train_data.append(instance)

print("Number of training instances:", len(train_data))
print("Number of testing instances:", len(test_data))
print("Total number of instances:", len(train_data) + len(test_data))

Number of training instances: 69
Number of testing instances: 19
Total number of instances: 88


### (c) Feature Extraction

#### i. Research

Some commonly used time-domain features for time series classification are:

- Minimum
- Maximum
- Mean
- Median
- Standard deviation
- Variance
- First quartile
- Third quartile
- Range
- Interquartile range
- Root mean square (RMS)
- Skewness
- Kurtosis

#### ii. Extraction

In [6]:
time_series_columns = [
    "avg_rss12",
    "var_rss12",
    "avg_rss13",
    "var_rss13",
    "avg_rss23",
    "var_rss23"
]

feature_rows = []

all_data = [
    ("train", instance) for instance in train_data
] + [
    ("test", instance) for instance in test_data
]

for i, (split, instance) in enumerate(all_data, start=1):
    data = instance["data"]

    row = {
        "Instance": i,
        "activity": instance["activity"],
        "dataset": instance["dataset"],
        "split": split
    }

    for j, column in enumerate(time_series_columns, start=1):
        series = data[column]

        row[f"min{j}"] = series.min()
        row[f"max{j}"] = series.max()
        row[f"mean{j}"] = series.mean()
        row[f"median{j}"] = series.median()
        row[f"std{j}"] = series.std()
        row[f"1st_quart{j}"] = series.quantile(0.25)
        row[f"3rd_quart{j}"] = series.quantile(0.75)

    feature_rows.append(row)

feature_df = pd.DataFrame(feature_rows)

feature_df.head()

feature_columns = [
    column for column in feature_df.columns
    if column not in ["Instance", "activity", "dataset", "split"]
]

print("Number of instances:", len(feature_df))
print("Number of extracted features:", len(feature_columns))
print("Shape of feature dataset:", feature_df.shape)

Number of instances: 88
Number of extracted features: 42
Shape of feature dataset: (88, 46)


#### iii. Standard Deviation

In [8]:
# Estimate the standard deviation and 90% bootstrap confidence interval
np.random.seed(42)

results = []

for feature in feature_columns:
    values = feature_df[feature].values
    
    # Standard deviation of the feature
    std_value = np.std(values, ddof=1)
    
    # Bootstrap
    bootstrap_stds = []
    
    for _ in range(10000):
        sample = np.random.choice(
            values,
            size=len(values),
            replace=True
        )
        
        bootstrap_stds.append(
            np.std(sample, ddof=1)
        )
    
    lower = np.percentile(bootstrap_stds, 5)
    upper = np.percentile(bootstrap_stds, 95)
    
    results.append([
        feature,
        std_value,
        lower,
        upper
    ])

std_ci_df = pd.DataFrame(
    results,
    columns=[
        "Feature",
        "Standard Deviation",
        "90% CI Lower",
        "90% CI Upper"
    ]
)

std_ci_df.round(3)

,Feature,Standard Deviation,90% CI Lower,90% CI Upper
0,min1,9.570,8.258,10.737
1,max1,4.394,3.334,5.305
2,mean1,5.336,4.712,5.880
3,median1,5.440,4.777,5.998
4,std1,1.772,1.571,1.952
5,1st_quart1,6.154,5.572,6.644
6,3rd_quart1,5.139,4.323,5.825
7,min2,0.000,0.000,0.000
8,max2,5.063,4.621,5.409
9,mean2,1.574,1.396,1.705


The standard deviation of each time-domain feature was estimated using all 88
instances. A bootstrap method with 10,000 resamples was used to construct the
90% confidence intervals. The 5th and 95th percentiles of the bootstrap
standard deviations were used as the lower and upper bounds. For min2 and min4, the estimated standard deviations are zero.

#### iv. Select Features

I selected minimum, mean, and maximum as the three most important time-domain features. Mean represents its average level. Minimum and maximum describe the range of the time series.These three features provide a simple summary of the distribution of each time series.

## 2. ISLR 3.7.4

### (a) Linear Train

The cubic regression will have a training RSS that is less than or equal to the linear regression. This is because the cubic model contains the linear model as a special case. It also has more flexibility. So,it can fit the training data at least as well as the linear model, even though the true relationship is linear.

### (b) Linear Test

The linear regression is expected to have a lower test RSS. Since the true relationship is linear, the cubic model adds unnecessary flexibility and may overfit the training data. So, the simpler linear model is expected to generalize better to new observations.

### (c) Not Linear Train

The cubic regression will have a training RSS that is less than or equal to
the linear regression. Since the cubic model is more flexible, it can fit the
training observations at least as well as the linear model.

### (d) Not Linear Testing

There is not enough information to determine which model will have the lower
test RSS. If the true relationship is sufficiently non-linear, the cubic model
may fit the relationship better and have a lower test RSS. If the
relationship is only slightly non-linear, the additional flexibility of the
cubic model may lead to overfitting. In this situation, the linear model may perform better.

## 3. ISLR 3.7.3 - Extra Practice 

## 3. ISLR 3.7.5 - Extra Practice 